In [ ]:
# Cell 1 — repository setup and all imports
import gc
import json
import os
from pathlib import Path
import subprocess
import sys

# Must be set before importing torch when deterministic CUDA execution is used.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")

import pandas as pd
import torch
from google.colab import drive
from IPython.display import display

REPO_URL = "https://github.com/VishR-94/dynamic_graphs_thesis.git"
REPO_REF = "main"
PROJECT_ROOT = Path("/content/dynamic_graphs_thesis")


def run_command(command, *, cwd=None):
    print("$", " ".join(str(part) for part in command))
    subprocess.run(command, cwd=cwd, check=True)


if not (PROJECT_ROOT / ".git").is_dir():
    run_command(
        [
            "git",
            "clone",
            "--recurse-submodules",
            "--branch",
            REPO_REF,
            REPO_URL,
            str(PROJECT_ROOT),
        ]
    )
else:
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=PROJECT_ROOT, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            "The Colab checkout contains uncommitted changes. Start a clean "
            "runtime before pulling the repository."
        )
    run_command(["git", "fetch", "origin", REPO_REF], cwd=PROJECT_ROOT)
    run_command(["git", "checkout", REPO_REF], cwd=PROJECT_ROOT)
    run_command(["git", "pull", "--ff-only", "origin", REPO_REF], cwd=PROJECT_ROOT)
    run_command(
        ["git", "submodule", "update", "--init", "--recursive"],
        cwd=PROJECT_ROOT,
    )

required_module = PROJECT_ROOT / "src" / "weather_benchmark" / "final_transfer.py"
if not required_module.is_file():
    raise RuntimeError(
        "The final-transfer module is missing. Apply and push the supplied "
        "all-city/all-test-year patch before running this notebook."
    )

run_command(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        str(PROJECT_ROOT / "requirements_weather_benchmark.txt"),
    ]
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.weather_benchmark.final_transfer import (
    CITY_DISPLAY_NAMES,
    FINAL_TRANSFER_CITIES,
    FINAL_TRANSFER_HORIZONS,
    FINAL_TRANSFER_TEST_YEARS,
    SELECTED_MODERN_TCN_ARCHITECTURES,
    audit_all_weather_city_csvs,
    build_city_test_metric_table,
    collect_selected_modern_tcn_transfer_metrics,
    preflight_selected_modern_tcn_architectures,
    run_selected_modern_tcn_transfer,
    save_selected_transfer_summaries,
    selected_transfer_plan,
)
from src.weather_benchmark.runner import ensure_weather_csv

print("Repository:", PROJECT_ROOT)
print(
    "Commit:",
    subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=PROJECT_ROOT, text=True
    ).strip(),
)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("CUBLAS_WORKSPACE_CONFIG:", os.environ.get("CUBLAS_WORKSPACE_CONFIG"))

In [ ]:
# Cell 2 — mount Google Drive
DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT))

# Fixed Graph-ModernTCN transfer across all Sonnet weather cities

The four horizon-specific architectures below were selected using the Hong Kong 2018 validation sweep. They are now frozen and retrained independently for each city and test year. There is no city-specific or year-specific hyperparameter tuning.

For test year `Y`, the executable Sonnet protocol uses validation year `Y-1` and training data from 1980 through `Y-2`. Every run still saves checkpoints, histories, scalers, priors, predictions, and final-context graph artifacts.

**Provenance note:** the architecture was selected using Hong Kong's 2017 validation set for the 2018 experiment. Applying it to 2016 and 2017 is therefore a retrospective fixed-architecture robustness test, not a temporally pristine hyperparameter-selection protocol for those earlier years. No weights, scalers, priors, or checkpoints are shared: every city/year model is retrained using its own causal Sonnet split.

In [ ]:
# Cell 3 — experiment and storage configuration
CITIES = tuple(FINAL_TRANSFER_CITIES)
TEST_YEARS = tuple(FINAL_TRANSFER_TEST_YEARS)
HORIZONS = tuple(FINAL_TRANSFER_HORIZONS)

# Colab representation of:
# /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/
# My Drive/dissertation/weather
OUTPUT_ROOT = Path("/content/drive/MyDrive/dissertation/weather")
DATA_CACHE_ROOT = Path("/content/sonnet_weather_data")
SUMMARY_ROOT = OUTPUT_ROOT / "final_selected_modernTCN_transfer"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE != "cuda":
    raise RuntimeError("Select a Colab GPU runtime before running the experiment cells.")

RESUME = True
OVERWRITE = False
SKIP_COMPLETED = True

# This stays True to preserve the existing selected Hong Kong 2018 run
# signatures and the complete artifact contract used by the sweep.
EXPORT_TRAIN_SPLIT = True

MAX_EPOCHS = 100
PATIENCE = 10
CONTINUE_ON_ERROR = False
DETERMINISTIC_RUNTIME = True

TRAIN_BATCH_SIZE = 16
VALIDATION_BATCH_SIZE = 32
EXPORT_BATCH_SIZE = 32
NUM_WORKERS = 0
PREFETCH_FACTOR = 2
PROGRESS_UPDATE_INTERVAL = 50

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
SUMMARY_ROOT.mkdir(parents=True, exist_ok=True)

print("Cities:", CITIES)
print("Test years:", TEST_YEARS)
print("Horizons:", HORIZONS)
print("Total expected runs:", len(CITIES) * len(TEST_YEARS) * len(HORIZONS))
print("Run artifact root:", OUTPUT_ROOT)
print("Summary root:", SUMMARY_ROOT)
print("Device:", DEVICE)

In [ ]:
# Cell 4 — display and save the locked four-architecture plan
architecture_rows = [
    {
        "horizon": horizon,
        "context_length": specification.context_length,
        "large_kernel": specification.large_kernel,
        "patch_size": specification.patch_size,
        "patch_stride": specification.patch_stride,
        "d_model": specification.d_model,
        "graph_hidden_dim": specification.graph_hidden_dim,
        "small_kernel": specification.small_kernel,
        "num_blocks": specification.num_blocks,
        "run_suffix": specification.run_suffix,
    }
    for horizon, specification in sorted(
        SELECTED_MODERN_TCN_ARCHITECTURES.items()
    )
]
architecture_table = pd.DataFrame(architecture_rows)
display(architecture_table)

experiment_plan = selected_transfer_plan(
    output_root=OUTPUT_ROOT,
    data_cache_root=DATA_CACHE_ROOT,
    cities=CITIES,
    test_years=TEST_YEARS,
    horizons=HORIZONS,
)
assert len(experiment_plan) == 60
assert experiment_plan["run_directory"].nunique() == 60
plan_path = SUMMARY_ROOT / "experiment_plan.csv"
experiment_plan.to_csv(plan_path, index=False)

display(experiment_plan)
print("Saved plan:", plan_path)
print(
    "The selected Hong Kong 2018 sweep directories use the same canonical "
    "suffixes. Completed matching runs will be signature-checked and skipped."
)

In [ ]:
# Cell 5 — download/audit every official city CSV
city_data_quality = audit_all_weather_city_csvs(
    cities=CITIES,
    data_cache_root=DATA_CACHE_ROOT,
)
quality_path = SUMMARY_ROOT / "city_data_quality.csv"
city_data_quality.to_csv(quality_path, index=False)
display(city_data_quality)
print("Saved city data-quality audit:", quality_path)

# Duplicate spatial nodes are retained rather than filtered because this final
# experiment intentionally runs the exact supplied Sonnet files for all cities.
for row in city_data_quality.itertuples(index=False):
    if row.exact_duplicate_node_pair_count:
        print(
            f"NOTE — {row.city_display_name}: exact duplicate node pairs: "
            f"{row.exact_duplicate_node_pairs}"
        )

In [ ]:
# Cell 6 — preflight the four frozen architectures
# Hong Kong 2018 is sufficient for tensor/parameter verification because all
# cities share the same [L, 9, 5] contract. Data/split manifests are saved per run.
PREFLIGHT_CITY = "hongkong"
PREFLIGHT_TEST_YEAR = 2018
PREFLIGHT_DATA_PATH = ensure_weather_csv(PREFLIGHT_CITY, DATA_CACHE_ROOT)

preflight = preflight_selected_modern_tcn_architectures(
    city=PREFLIGHT_CITY,
    test_year=PREFLIGHT_TEST_YEAR,
    data_path=PREFLIGHT_DATA_PATH,
    output_root=OUTPUT_ROOT,
    project_root=PROJECT_ROOT,
    horizons=HORIZONS,
    device=DEVICE,
    train_batch_size=TRAIN_BATCH_SIZE,
    validation_batch_size=VALIDATION_BATCH_SIZE,
    export_batch_size=EXPORT_BATCH_SIZE,
    progress_update_interval=PROGRESS_UPDATE_INTERVAL,
    prefetch_factor=PREFETCH_FACTOR,
    deterministic_runtime=DETERMINISTIC_RUNTIME,
)

preflight_display = pd.DataFrame(
    {
        "H": row.horizon,
        "L": row.context_length,
        "kernel": row.modern_tcn_large_kernel,
        "patch_stride": row.modern_tcn_patch_stride,
        "d_model": row.modern_tcn_d_model,
        "graph_hidden_dim": row.modern_tcn_graph_hidden_dim,
        "train_batch": row.train_batch_size,
        "train_windows": row.window_counts["train"],
        "validation_windows": row.window_counts["validation"],
        "test_windows": row.window_counts["test"],
        "input_shape": str(row.input_shape),
        "prediction_shape": str(row.prediction_shape),
        "saved_graph_shapes": str(row.saved_final_graph_shapes),
        "trainable_parameters": row.parameter_counts["trainable_parameters"],
        "run_directory": row.run_directory,
    }
    for row in preflight.itertuples(index=False)
)
display(preflight_display)
assert len(preflight_display) == 4
print("All four architecture preflights passed.")

del preflight
gc.collect()
torch.cuda.empty_cache()

## Run all 60 fixed experiments

This cell is resumable. Completed runs are skipped. An interrupted run resumes from its last end-of-epoch checkpoint. The four selected Hong Kong 2018 sweep runs are reused only when their saved experiment signatures match exactly.

In [ ]:
# Cell 7 — train/evaluate all cities × test years × horizons
training_summary = run_selected_modern_tcn_transfer(
    output_root=OUTPUT_ROOT,
    data_cache_root=DATA_CACHE_ROOT,
    project_root=PROJECT_ROOT,
    summary_directory=SUMMARY_ROOT,
    cities=CITIES,
    test_years=TEST_YEARS,
    horizons=HORIZONS,
    device=DEVICE,
    resume=RESUME,
    overwrite=OVERWRITE,
    skip_completed=SKIP_COMPLETED,
    export_train_split=EXPORT_TRAIN_SPLIT,
    max_epochs=MAX_EPOCHS,
    patience=PATIENCE,
    num_workers=NUM_WORKERS,
    continue_on_error=CONTINUE_ON_ERROR,
    train_batch_size=TRAIN_BATCH_SIZE,
    validation_batch_size=VALIDATION_BATCH_SIZE,
    export_batch_size=EXPORT_BATCH_SIZE,
    progress_update_interval=PROGRESS_UPDATE_INTERVAL,
    prefetch_factor=PREFETCH_FACTOR,
    deterministic_runtime=DETERMINISTIC_RUNTIME,
)

display(training_summary)
print("Status counts:")
display(training_summary["status"].value_counts(dropna=False).rename("runs"))

In [ ]:
# Cell 8 — collect saved metrics and verify expected artifacts
all_metrics = collect_selected_modern_tcn_transfer_metrics(
    output_root=OUTPUT_ROOT,
    cities=CITIES,
    test_years=TEST_YEARS,
    horizons=HORIZONS,
)

status_columns = [
    "city",
    "test_year",
    "horizon",
    "status",
    "architecture_match",
    "best_epoch",
    "best_validation_score",
    "test_mae",
    "test_r",
    "test_smape",
    "missing_artifact_count",
    "run_directory",
]
display(all_metrics[status_columns])

incomplete = all_metrics.loc[
    ~all_metrics["status"].eq("completed")
    | all_metrics["missing_artifact_count"].ne(0)
]
if incomplete.empty:
    print("All 60 runs are complete and all required artifacts are present.")
else:
    print(f"Incomplete or invalid rows: {len(incomplete)}")
    display(
        incomplete[
            [
                "city",
                "test_year",
                "horizon",
                "status",
                "architecture_error",
                "missing_artifacts",
                "run_directory",
            ]
        ]
    )

# Final test metrics by location

Each table has four horizon rows. The three top-level column groups are the independently retrained 2016, 2017, and 2018 test splits; each contains MAE, linear correlation `r`, and weather sMAPE at the central node and final forecast position.

In [ ]:
# Cell 9 — display one multi-level metrics table per city and save summaries
summary_paths = save_selected_transfer_summaries(
    metrics=all_metrics,
    output_root=OUTPUT_ROOT,
    summary_directory=SUMMARY_ROOT,
    cities=CITIES,
    test_years=TEST_YEARS,
    horizons=HORIZONS,
)

city_tables = {}
for city in CITIES:
    table = build_city_test_metric_table(
        all_metrics,
        city=city,
        test_years=TEST_YEARS,
        horizons=HORIZONS,
    )
    city_tables[city] = table
    formatters = {
        column: ("{:.2f}" if column[1] == "sMAPE" else "{:.4f}")
        for column in table.columns
    }
    print("\n" + "=" * 96)
    print(CITY_DISPLAY_NAMES[city])
    display(
        table.style
        .format(formatters, na_rep="—")
        .set_caption(
            f"{CITY_DISPLAY_NAMES[city]} — central T850 final-horizon test metrics"
        )
    )

print("\nSaved summary artifacts:")
for name, path in summary_paths.items():
    print(f"{name}: {path}")

In [ ]:
# Cell 10 — final completion gate
expected_runs = len(CITIES) * len(TEST_YEARS) * len(HORIZONS)
completed_runs = int(all_metrics["status"].eq("completed").sum())
valid_architectures = int(all_metrics["architecture_match"].eq(True).sum())
artifact_complete = int(all_metrics["missing_artifact_count"].eq(0).sum())

print("Expected runs:", expected_runs)
print("Completed runs:", completed_runs)
print("Architecture-matched runs:", valid_architectures)
print("Runs with complete required artifacts:", artifact_complete)

assert completed_runs == expected_runs, "Some final-transfer runs are incomplete."
assert valid_architectures == expected_runs, "An exported run has the wrong architecture."
assert artifact_complete == expected_runs, "Some required artifacts are missing."

print("Final all-city/all-test-year transfer is complete.")